# Racing Line Chart of Analytic Tools

## Import Libraries

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import pandas as pd

my_colors = plt.get_cmap("tab20").colors
my_colors = {
    "Excel": "#5D7F24",
    "Minitab": "#00A9E0",
    "R": "#DD0511",
    "SAS": "#FFB81C",
    "Adobe Flash": "#FF0000",
    "Python": "#ff8968",
    "Tableau": "#76ACC1",
    "Power BI": "#FFC907",
    "Alteryx": "#FF6F61",
    "GenAI": "#BB84A5",
    "SQL": "#FF7F50",
    "Agentic AI": "#0B4F6C",
}

In [ ]:
from PIL import Image

df = pd.read_excel("ronsAnalyticsToolsOverTime.xlsx")
my_colors = plt.get_cmap("tab20").colors
tools = df.columns[3:]

FULL_HISTORY_START = df["Date"].min()
FINAL_DATE = df["Date"].max()

# --- Camera-zoom finale settings ---
# Pan the left edge of the x-axis from the full history down to a fixed
# 5-year window, finishing exactly on the last frame. The pan itself starts
# ZOOM_LEAD_YEARS before the end so it's a gentle, gradual push-in rather
# than a sudden jump.
ZOOM_WINDOW_YEARS = 5
ZOOM_LEAD_YEARS = 8
ZOOM_TARGET_START = FINAL_DATE - pd.DateOffset(years=ZOOM_WINDOW_YEARS)
ZOOM_PAN_START_DATE = FINAL_DATE - pd.DateOffset(years=ZOOM_LEAD_YEARS)

# How much longer to hold the final (zoomed-in) frame, in seconds.
HOLD_SECONDS = 10


def ease_smoothstep(p):
    p = max(0.0, min(1.0, p))
    return p * p * (3 - 2 * p)


fig, ax = plt.subplots(figsize=(10, 6))


def update(frame):
    ax.clear()
    # 1. Turn off spines (the borders)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)

    subset = df.iloc[: frame + 1]

    # Calculate dynamic limits
    current_ymax = subset[tools].max().max()
    if pd.isna(current_ymax) or current_ymax == 0:
        current_ymax = 10

    # 2. Plotting Lines
    for i, tool in enumerate(tools):
        ax.plot(subset["Date"], subset[tool], label=tool, color=my_colors[i], lw=2)

        if not subset.empty:
            last_row = subset.iloc[-1]
            # Use ha='left' and a small offset to keep labels away from the dots
            ax.scatter(last_row["Date"], last_row[tool], color=my_colors[i], s=40)
            ax.text(
                last_row["Date"],
                last_row[tool],
                f"  {tool}",
                va="center",
                ha="left",
                fontsize=9,
                fontweight="bold",
            )

    # 3. Handle Event Lines (Horizontal labels at current peak)
    event_indices = subset[subset["ShortEvent"].notna()].index
    for idx in event_indices:
        e_date, e_name = df.loc[idx, "Date"], df.loc[idx, "ShortEvent"]
        e_max = df.iloc[idx][tools].max()
        label_y = e_max * 1.08
        ax.vlines(
            x=e_date, ymin=0, ymax=label_y, color="black", linestyle="--", alpha=1
        )
        ax.text(x=e_date, y=label_y, s=e_name, ha="center", va="bottom", fontsize=8)

    # 4. Relocate Legend to the LEFT inside the plot
    # 'upper left' with a slight inset looks very clean
    ax.legend(loc="upper left", frameon=False, fontsize="small", labelspacing=0.5)

    # 5. X-Axis: expand through history, then camera-zoom into the last
    # ZOOM_WINDOW_YEARS years as we approach the final frame.
    if not subset.empty:
        current_date = subset["Date"].max()

        pan_span = (FINAL_DATE - ZOOM_PAN_START_DATE).days
        progress = (
            (current_date - ZOOM_PAN_START_DATE).days / pan_span if pan_span else 1.0
        )
        p = ease_smoothstep(progress)

        xmin = FULL_HISTORY_START + p * (ZOOM_TARGET_START - FULL_HISTORY_START)
        # Right-side buffer shrinks from 18 months down to 6 as we zoom in,
        # so the finale doesn't waste space on empty future dates.
        buffer_days = 548 - p * (548 - 182)
        xmax = current_date + pd.Timedelta(days=buffer_days)
        ax.set_xlim(xmin, xmax)

    # 6. Final Polish
    ax.set_ylim(0, current_ymax * 1.3)
    ax.set_yticklabels([])  # Keep y-axis clean
    ax.set_yticks([])
    ax.set_title(
        "Ron's Data Analytic Tool Usage Over Time", loc="left", pad=20, fontsize=14
    )
    plt.xticks(rotation=45)

    # Manual margin control - giving the right side plenty of room
    plt.subplots_adjust(right=0.85, left=0.05, top=0.9, bottom=0.15)


# Save the animation
ani = FuncAnimation(fig, update, frames=len(df), interval=300)
ani.save("racing_line_chart_with_labels.gif", writer="pillow", fps=5)
plt.show()

# Hold the final (zoomed-in) frame a few extra seconds so viewers have time
# to read it before the gif loops. We do this as a post-processing pass on
# the saved gif rather than by repeating the last frame index above,
# because Pillow's GIF writer silently collapses identical consecutive
# frames, which would otherwise erase the hold.
with Image.open("racing_line_chart_with_labels.gif") as gif:
    frames, durations = [], []
    for i in range(gif.n_frames):
        gif.seek(i)
        frames.append(gif.convert("RGBA"))
        durations.append(gif.info.get("duration", 200))
    durations[-1] += HOLD_SECONDS * 1000
    frames[0].save(
        "racing_line_chart_with_labels.gif",
        save_all=True,
        append_images=frames[1:],
        duration=durations,
        loop=0,
        disposal=2,
    )